
# IR Book Subcorpora — Compound-Term Pipeline (Regenerated)
Focuses on **compound IR terms** (e.g., *regional security complex*, *balance of power*, *structural realism*).
- Loads chapter-wise corpora
- TF–IDF (words + compound phrases)
- NMF topics & KMeans clusters (compound phrases)
- Chapter summary tables
- Outputs to `./subcorpora_outputs`


In [1]:

# === CONFIG ===
from pathlib import Path
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
BASE_DIR = PROJECT_ROOT / "archive" / "output_dump" / "book3_subcorpora"
OUT_DIR = PROJECT_ROOT / "data" / "output" / "analysis"
N_TOPICS = 5
N_CLUSTERS = 5
NGRAM_RANGE = (1, 3)
MIN_PARA_WORDS = 8

Path(OUT_DIR).mkdir(exist_ok=True, parents=True)
print("Using BASE_DIR =", BASE_DIR)


Using BASE_DIR = archive/output_dump/book3_subcorpora


In [2]:

# === IMPORTS ===
import os, re, json, numpy as np, pandas as pd
from pathlib import Path

# Ensure deps
try:
    import nltk
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk"])
    import nltk

try:
    import sklearn
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    import sklearn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# NLTK resources
for pkg in ["punkt", "stopwords"]:
    try:
        nltk.data.find(f"corpora/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)
for pkg in ["averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.data.find(f"taggers/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words("english"))
from nltk import pos_tag, word_tokenize


In [3]:

# === LOADING & BASIC TABLES ===
def normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

def load_chapter_dirs(base_dir: str):
    base = Path(base_dir)
    if not base.exists():
        raise FileNotFoundError(f"Directory not found: {base_dir}")
    return sorted(p for p in base.iterdir() if p.is_dir() and p.name.startswith("chapter_"))

def read_chapter(ch_dir: Path):
    meta_path = ch_dir / "meta.json"
    corpus_path = ch_dir / "corpus.txt"
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
    text = corpus_path.read_text(encoding="utf-8") if corpus_path.exists() else ""
    paras = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return meta, paras

rows_paras, rows_ch = [], []
for ch_dir in load_chapter_dirs(BASE_DIR):
    meta, paras = read_chapter(ch_dir)
    ch_id = ch_dir.name
    title = meta.get("title", ch_id)
    n_words = sum(len(p.split()) for p in paras)
    rows_ch.append({
        "chapter": ch_id, "title": title,
        "n_paragraphs": len(paras), "n_words": n_words,
        "avg_words_per_para": (n_words/len(paras)) if paras else 0.0
    })
    for i, p in enumerate(paras, 1):
        rows_paras.append({"chapter": ch_id, "para_id": i, "text": p, "word_count": len(p.split())})

df_chapters = pd.DataFrame(rows_ch).sort_values("chapter").reset_index(drop=True)
df_paras = pd.DataFrame(rows_paras).sort_values(["chapter","para_id"]).reset_index(drop=True)

df_chapters.to_csv(Path(OUT_DIR)/"chapters_overview.csv", index=False)
df_paras.to_csv(Path(OUT_DIR)/"paragraphs_raw.csv", index=False)

print("Chapters:", df_chapters.shape, "| Paragraphs:", df_paras.shape)
df_chapters.head(10)


Chapters: (5, 5) | Paragraphs: (618, 4)


,chapter,title,n_paragraphs,n_words,avg_words_per_para
0,chapter_01,2. STATUS OF REGIONAL TRANSPORT,80,11158,139.475000
1,chapter_02,3. Main Regional Transport Issues,51,5195,101.862745
2,chapter_03,4. Action Plan,57,13938,244.526316
3,chapter_04,5. Suggested ADB Road Map,17,3693,217.235294
4,chapter_05,6. Annexures,413,9819,23.774818


In [4]:

# === BASELINE WORD-LEVEL TF–IDF ===
def clean_text(s: str):
    s = s.lower()
    s = re.sub(r"[\-–—]", " ", s)
    s = re.sub(r"[^a-z0-9' ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df_paras["clean"] = df_paras["text"].apply(clean_text)
df_paras["keep"]  = df_paras["word_count"].ge(MIN_PARA_WORDS)
df_work = df_paras[df_paras["keep"]].copy()

def top_tfidf_terms(texts, n_terms=20):
    if not texts: return []
    vec = TfidfVectorizer(ngram_range=NGRAM_RANGE, min_df=2)
    X = vec.fit_transform(texts)
    if X.shape[1] == 0: return []
    scores = np.asarray(X.mean(axis=0)).ravel()
    terms = np.array(vec.get_feature_names_out())
    idx = np.argsort(-scores)[:n_terms]
    return list(zip(terms[idx], scores[idx]))

rows = []
for ch, sub in df_work.groupby("chapter", sort=False):
    terms = top_tfidf_terms(sub["clean"].tolist(), n_terms=25)
    for t, s in terms:
        rows.append({"chapter": ch, "term": t, "score": float(s)})
df_terms_words = pd.DataFrame(rows)
df_terms_words.to_csv(Path(OUT_DIR)/"chapter_top_terms_words.csv", index=False)
df_terms_words.head(10)


,chapter,term,score
0,chapter_01,the,0.146679
1,chapter_01,strategy,0.125533
2,chapter_01,of,0.106431
3,chapter_01,in,0.082703
4,chapter_01,and,0.082047
5,chapter_01,to,0.069072
6,chapter_01,draft strategy,0.065094
7,chapter_01,central asia reassessment,0.065094
8,chapter_01,strategy draft strategy,0.065094
9,chapter_01,reassessment of the,0.065094


In [5]:

# === COMPOUND PHRASE EXTRACTOR (POS-based) ===
ALLOWED_POS = {"NN","NNS","NNP","NNPS","JJ"}
PATTERNS = {
    ("JJ","NN"), ("NN","NN"),
    ("JJ","JJ","NN"), ("JJ","NN","NN"), ("NN","JJ","NN"), ("NN","NN","NN")
}

def normalize_phrase(tokens):
    toks = [re.sub(r"[^a-z0-9\-]+","", t.lower()) for t in tokens]
    toks = [t for t in toks if t and not re.fullmatch(r"\d+", t)]
    return " ".join(toks)

def extract_compound_phrases(text):
    phrases = []
    tokens = word_tokenize(text)
    if not tokens: return phrases
    tagged = pos_tag(tokens)
    for n in (2,3):
        for i in range(len(tagged)-n+1):
            window = tagged[i:i+n]
            pos_seq = tuple(tag for _, tag in window)
            if not all(p in ALLOWED_POS for p in pos_seq): continue
            if pos_seq not in PATTERNS: continue
            surf = [tok for tok, _ in window]
            phrase = normalize_phrase(surf)
            if len(phrase.split()) == n and all(len(w) >= 2 for w in phrase.split()):
                phrases.append(phrase)
    return phrases


In [6]:

# === GLOBAL COMPOUND VOCAB + VECTORIZER ===
from collections import Counter

all_para_phrases = [extract_compound_phrases(t) for t in df_paras["text"].tolist()]
phrase_counts = Counter(p for plist in all_para_phrases for p in plist)

def build_vocab(min_count):
    kept = [p for p,c in phrase_counts.items() if c >= min_count]
    return {p:i for i,p in enumerate(sorted(kept))}

VOCAB = build_vocab(min_count=2)
if len(VOCAB) == 0:
    VOCAB = build_vocab(min_count=1)

print(f"[INFO] Compound vocab size: {len(VOCAB)}")

def encode_doc_from_phrases(phrases):
    return " || ".join(phrases) if phrases else ""

df_paras["phrases"] = all_para_phrases
df_paras["doc_encoded"] = df_paras["phrases"].apply(encode_doc_from_phrases)

def make_vec_from_vocab(vocab):
    return TfidfVectorizer(
        vocabulary=vocab,
        analyzer="word",
        preprocessor=lambda s: s,
        tokenizer=lambda s: s.split(" || "),
        token_pattern=None,
        lowercase=False,
        norm="l2",
        use_idf=True,
        smooth_idf=True,
        sublinear_tf=False,
    )

vec_comp = make_vec_from_vocab(VOCAB)


[INFO] Compound vocab size: 329


In [7]:

# === TF–IDF (COMPOUND PHRASES) ===
rows = []
for ch_id, sub in df_paras.groupby("chapter", sort=False):
    docs = sub["doc_encoded"].tolist()
    if not any(docs): continue
    X = vec_comp.fit_transform(docs)
    if X.shape[1] == 0: continue
    scores = np.asarray(X.mean(axis=0)).ravel()
    terms = np.array(vec_comp.get_feature_names_out())
    top = np.argsort(-scores)[:25]
    for idx in top:
        rows.append({"chapter": ch_id, "term": terms[idx], "score": float(scores[idx])})

df_terms_comp = pd.DataFrame(rows)
df_terms_comp.to_csv(Path(OUT_DIR)/"chapter_top_terms_compound.csv", index=False)
df_terms_comp.head(10)


,chapter,term,score
0,chapter_01,road transport,0.030868
1,chapter_01,transit traffic,0.025469
2,chapter_01,international traffic,0.020596
3,chapter_01,regional transport,0.018462
4,chapter_01,core network,0.015113
5,chapter_01,private sector,0.014688
6,chapter_01,legal framework,0.013109
7,chapter_01,na na,0.012500
8,chapter_01,ferry service,0.012500
9,chapter_01,iranian network,0.012334


In [10]:
# --- Robust NMF on compound phrases (retry + adaptive topics) ---
import numpy as np
import warnings
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning

def pick_n_topics(X, target=5, cap=12):
    n_docs = X.shape[0]
    # keep topics reasonable for small chapters
    auto = max(2, min(target, int(np.sqrt(max(2, n_docs))) + 1))
    return min(auto, cap, n_docs - 1) if n_docs > 2 else 2

def nmf_fit_robust(X, terms, target_topics=5):
    if X.shape[0] < 2 or X.shape[1] < 2:
        return []

    n_topics = pick_n_topics(X, target=target_topics)

    # Try (Frobenius, CD) first, then fallback to (KL, MU)
    trials = [
        dict(beta_loss="frobenius", solver="cd", max_iter=1500, tol=1e-4),
        dict(beta_loss="kullback-leibler", solver="mu", max_iter=2000, tol=1e-4),
    ]
    for cfg in trials:
        try:
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", category=ConvergenceWarning)
                nmf = NMF(
                    n_components=n_topics,
                    init="nndsvda",
                    random_state=42,
                    **cfg
                )
                W = nmf.fit_transform(X)
                H = nmf.components_
            # Build topic word lists
            topics = []
            for k in range(nmf.n_components):
                top_idx = np.argsort(-H[k])[:8]
                topics.append(", ".join(terms[top_idx]))
            return topics
        except Exception as e:
            # Try next config
            continue
    return []  # if both attempts fail

# Use the robust NMF in your loop
topics_compound = []
for ch_id, sub in df_paras.groupby("chapter", sort=False):
    docs = sub["doc_encoded"].tolist()
    if not any(docs):
        continue
    X = vec_comp.fit_transform(docs)
    terms = np.array(vec_comp.get_feature_names_out())

    topics = nmf_fit_robust(X, terms, target_topics=N_TOPICS)
    for ti, words in enumerate(topics):
        topics_compound.append({"chapter": ch_id, "topic": int(ti), "words": words})

df_topics_comp = pd.DataFrame(topics_compound)
df_topics_comp.to_csv(Path(OUT_DIR)/"chapter_topics_nmf_compound.csv", index=False)
print("[OK] topics:", df_topics_comp.shape)
df_topics_comp.head(10)


[OK] topics: (25, 3)


,chapter,topic,words
0,chapter_01,0,"road transport, international road, internatio..."
1,chapter_01,1,"transit traffic, total traffic, dominant role,..."
2,chapter_01,2,"trunk line, main trunk, main trunk line, irani..."
3,chapter_01,3,"international traffic, core network, rail netw..."
4,chapter_01,4,"road network, republic road, road condition, r..."
5,chapter_02,0,"road transport, international road transport, ..."
6,chapter_02,1,"road transport, trade facilitation, truck km, ..."
7,chapter_02,2,"private sector, regional transport, physical i..."
8,chapter_02,3,"inadequate maintenance, trade facilitation, co..."
9,chapter_02,4,"road maintenance, annualised basis, road user,..."


In [11]:

# === CHAPTER SUMMARIES ===
title_map = dict(zip(df_chapters["chapter"], df_chapters["title"]))

# Words summary
rows = []
for ch in df_chapters["chapter"]:
    top5 = df_terms_words[df_terms_words["chapter"]==ch].sort_values("score", ascending=False)["term"].head(5).tolist()
    rows.append({"Chapter": title_map.get(ch, ch), "Top_Terms_Words": ", ".join(top5)})
df_summary_words = pd.DataFrame(rows)
df_summary_words.to_csv(Path(OUT_DIR)/"chapter_summary_words.csv", index=False)

# Compound summary
rows = []
for ch in df_chapters["chapter"]:
    top5 = df_terms_comp[df_terms_comp["chapter"]==ch].sort_values("score", ascending=False)["term"].head(5).tolist()
    topics_sub = df_topics_comp[df_topics_comp["chapter"]==ch].head(3)
    topics_strs = [f"Topic {int(t['topic'])+1}: {t['words']}" for _, t in topics_sub.iterrows()]
    rows.append({"Chapter": title_map.get(ch, ch),
                 "Top_Compound_Terms": ", ".join(top5),
                 "Compound_Topics": " | ".join(topics_strs)})
df_summary_comp = pd.DataFrame(rows)
df_summary_comp.to_csv(Path(OUT_DIR)/"chapter_summary_compound.csv", index=False)

print("Saved summaries to", OUT_DIR)
df_summary_comp.head(10)


Saved summaries to data/output/analysis


,Chapter,Top_Compound_Terms,Compound_Topics
0,2. STATUS OF REGIONAL TRANSPORT,"road transport, transit traffic, international...","Topic 1: road transport, international road, i..."
1,3. Main Regional Transport Issues,"road transport, regional transport, regional r...","Topic 1: road transport, international road tr..."
2,4. Action Plan,"regional transport, road transport, regional r...","Topic 1: road transport, regional road transpo..."
3,5. Suggested ADB Road Map,"feasibility study, road transport, road map, r...","Topic 1: feasibility study, second phase, regi..."
4,6. Annexures,"external debt, growth rate, debt service, exte...","Topic 1: external debt, economic growth, forei..."
